# Capítulo 5 · Proyecto 1: comparación de modelos

> ¿Qué modelo recomendarías para clasificar la presencia de enfermedad cardiaca y qué resultados respaldan tu elección?

Utiliza el dataset [UCI Heart Disease de Kaggle](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data), archivo `heart_disease_uci.csv`, y documenta tu trabajo en un notebook reproducible.

## Etapa 1: limpieza y preparación

1. Identifica los valores faltantes y los valores inválidos, decide cómo tratarlos y justifica brevemente tu decisión.
2. Explora los posibles outliers y explica si los conservarás o aplicarás algún tratamiento.
3. Prepara las variables numéricas y categóricas. Crea el objetivo binario (**target**) a partir de `num`: **target = 0 si num = 0** (ausencia de enfermedad) y **target = 1 si num es 1, 2, 3 o 4** (presencia de enfermedad). Excluye tanto `num` como el target de las entradas del modelo.

## Etapa 2: entrenamiento y comparación

1. Separa los datos en entrenamiento, validación y prueba, usando las mismas particiones para todos los modelos y `random_state=42`. Ajusta la imputación y el escalado únicamente con entrenamiento.
2. Entrena **regresión logística, KNN y árbol de decisión**, aplicando escalado cuando corresponda.
3. Usa **GridSearchCV** con validación cruzada sobre los datos de entrenamiento para buscar los mejores hiperparámetros de los tres modelos. Compara las mejores configuraciones en validación y selecciona el modelo con mejor resultado según una métrica justificada.
4. Compara entrenamiento y evaluación para identificar sobreajuste. Presenta una tabla de métricas y las matrices de confusión de prueba.
5. Recomienda un modelo y justifica tu elección con los resultados: qué errores comete, qué ventajas tiene y cuáles son sus limitaciones.

| Nombre original | Significado en español |
|---|---|
| `id` | Identificador del registro |
| `age` | Edad |
| `sex` | Sexo |
| `dataset` | Origen de los datos |
| `cp` | Tipo de dolor torácico |
| `trestbps` | Presión arterial en reposo |
| `chol` | Colesterol |
| `fbs` | Glucosa en ayunas superior a 120 mg/dL |
| `restecg` | Resultado del electrocardiograma en reposo |
| `thalch` | Frecuencia cardíaca máxima alcanzada |
| `exang` | Angina provocada por el ejercicio |
| `oldpeak` | Descenso del segmento ST durante el ejercicio respecto al reposo |
| `slope` | Pendiente del segmento ST durante el máximo esfuerzo |
| `ca` | Número de vasos principales observados por fluoroscopia |
| `thal` | Resultado de la prueba de talio |
| `num` | Diagnóstico: 0 indica ausencia de enfermedad; 1–4 indican presencia |

## Descarga de Dataset

In [7]:
!pip install kagglehub 


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\robof\.cache\kagglehub\datasets\redwankarimsony\heart-disease-data\versions\6


## Visualización del dataset

In [9]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, roc_curve
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(raiz / "data" / "heart_disease_uci.csv")
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [ ]:
print ("Filas y columnas del dataset: ", df.shape)
df.info()

Filas y columnas del dataset:  (920, 16)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


## Visualización de nulos

In [16]:
# Valores faltantes: cantidad y porcentaje por columna
pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(1),
})

,nulos,porcentaje
id,0,0.0
age,0,0.0
sex,0,0.0
dataset,0,0.0
cp,0,0.0
trestbps,59,6.4
chol,30,3.3
fbs,90,9.8
restecg,2,0.2
thalch,55,6.0


## Filas con valores faltantes

In [17]:
# Filas con algún valor faltante (axis=1: ¿hay algún NaN en la fila?)
df[df.isna().any(axis=1)].head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num,target
87,88,53,Female,Cleveland,non-anginal,128.0,216.0,False,lv hypertrophy,115.0,False,0.0,upsloping,0.0,NaN,0,0
166,167,52,Male,Cleveland,non-anginal,138.0,223.0,False,normal,169.0,False,0.0,upsloping,NaN,normal,0,0
192,193,43,Male,Cleveland,asymptomatic,132.0,247.0,True,lv hypertrophy,143.0,True,0.1,flat,NaN,reversable defect,1,1
266,267,52,Male,Cleveland,asymptomatic,128.0,204.0,True,normal,156.0,True,1.0,flat,0.0,NaN,2,1
287,288,58,Male,Cleveland,atypical angina,125.0,220.0,False,normal,144.0,False,0.4,flat,NaN,reversable defect,0,0


## Visualización de duplicados

In [18]:
print("Filas duplicadas:", df.duplicated().sum())

Filas duplicadas: 0


In [19]:
df.describe()

,id,age,trestbps,chol,thalch,oldpeak,ca,num,target
count,920.000000,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000,920.000000
mean,460.500000,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652,0.553261
std,265.725422,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693,0.497426
min,1.000000,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000,0.000000
25%,230.750000,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000,0.000000
50%,460.500000,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000,1.000000
75%,690.250000,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000,1.000000
max,920.000000,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000,1.000000


In [15]:
"""import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Creamos el target binario que pide el proyecto (si no lo has creado aún)
df['target'] = (df['num'] > 0).astype(int)

# 2. Seleccionamos solo las variables numéricas de interés (excluyendo 'id')
columnas_numericas = df.select_dtypes(include=[np.number]).columns.drop('id', errors='ignore')
corr_matrix = df[columnas_numericas].corr()

# 3. Creamos una máscara triangular para no repetir la información simétrica
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 4. Graficamos el mapa de calor
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8}
)

plt.title("Matriz de Correlación de Variables Numéricas", fontsize=14, pad=15)
plt.tight_layout()
plt.show()"""

'import numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# 1. Creamos el target binario que pide el proyecto (si no lo has creado aún)\ndf[\'target\'] = (df[\'num\'] > 0).astype(int)\n\n# 2. Seleccionamos solo las variables numéricas de interés (excluyendo \'id\')\ncolumnas_numericas = df.select_dtypes(include=[np.number]).columns.drop(\'id\', errors=\'ignore\')\ncorr_matrix = df[columnas_numericas].corr()\n\n# 3. Creamos una máscara triangular para no repetir la información simétrica\nmask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n\n# 4. Graficamos el mapa de calor\nplt.figure(figsize=(10, 8))\nsns.heatmap(\n    corr_matrix,\n    mask=mask,\n    annot=True,\n    fmt=".2f",\n    cmap="coolwarm",\n    vmin=-1,\n    vmax=1,\n    linewidths=0.5,\n    square=True,\n    cbar_kws={"shrink": 0.8}\n)\n\nplt.title("Matriz de Correlación de Variables Numéricas", fontsize=14, pad=15)\nplt.tight_layout()\nplt.show()'